# GCP Collector — Edmonton Retail Map Georeferencing

Fluxo:
1. Upload do PDF/AI (ou imagem) do mapa base
2. Clica nos pontos de interesse no mapa (fica marcado com número)
3. Preenche o lat/lng real de cada ponto clicado (Google Maps)
4. Exporta CSV com: nome, x_pdf, y_pdf, lat, lng


## 1. Setup

In [ ]:
!pip install PyMuPDF ipycanvas ipywidgets --quiet

import fitz  # PyMuPDF
from ipycanvas import Canvas, hold_canvas
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
import io
from PIL import Image
import numpy as np
import csv

## 2. Upload do arquivo (PDF, AI, ou imagem)
Se for `.ai`/`.pdf`, renderiza a primeira página em alta resolução.
Se for `.png`/`.jpg`, usa direto.

In [ ]:
uploaded = files.upload()
fname = list(uploaded.keys())[0]
print('Uploaded:', fname)

RENDER_SCALE = 3  # zoom factor when rasterizing PDF/AI, keep same value used later for coord conversion

if fname.lower().endswith(('.pdf', '.ai')):
    doc = fitz.open(stream=uploaded[fname], filetype='pdf')
    page = doc[0]
    pdf_w, pdf_h = page.rect.width, page.rect.height
    mat = fitz.Matrix(RENDER_SCALE, RENDER_SCALE)
    pix = page.get_pixmap(matrix=mat)
    img = Image.open(io.BytesIO(pix.tobytes('png'))).convert('RGB')
    print(f'PDF page size: {pdf_w} x {pdf_h} pts -> rendered {img.width} x {img.height} px')
else:
    img = Image.open(io.BytesIO(uploaded[fname])).convert('RGB')
    pdf_w, pdf_h = img.width, img.height  # 1:1, no pdf coord system
    RENDER_SCALE = 1
    print(f'Image size: {img.width} x {img.height} px')

img_arr = np.array(img)
print('Ready.')

## 3. Mapa interativo — clique para marcar pontos
- Clique no mapa em cada interseção que você reconhece.
- Cada clique gera uma linha na tabela abaixo, com um campo pra você digitar o nome e o lat/lng.
- Se clicar errado, use o botão "Remover último ponto".
- Dá pra dar zoom mudando `DISPLAY_SCALE` abaixo e rodando de novo (útil se a imagem for muito grande pra tela).

In [ ]:
# Ajuste isso se a imagem for maior que a tela - reduz o tamanho de EXIBIÇÃO sem perder a resolução original
MAX_DISPLAY_WIDTH = 1000
display_scale = min(1.0, MAX_DISPLAY_WIDTH / img.width)
disp_w, disp_h = int(img.width*display_scale), int(img.height*display_scale)
disp_img = img.resize((disp_w, disp_h))

canvas = Canvas(width=disp_w, height=disp_h)
canvas.put_image_data(np.array(disp_img), 0, 0)

points = []  # each entry: {'name':..., 'x_pdf':..., 'y_pdf':..., 'lat':None, 'lng':None}
output_area = widgets.Output()

def redraw():
    with hold_canvas(canvas):
        canvas.put_image_data(np.array(disp_img), 0, 0)
        canvas.stroke_style = 'red'
        canvas.fill_style = 'red'
        canvas.font = '14px sans-serif'
        for i, p in enumerate(points):
            dx = p['x_pdf'] * display_scale * RENDER_SCALE
            dy = p['y_pdf'] * display_scale * RENDER_SCALE
            canvas.stroke_circle(dx, dy, 6)
            canvas.fill_text(str(i+1), dx+8, dy-8)

def handle_click(x, y):
    # x,y are in displayed-canvas pixel space -> convert back to PDF point space
    x_pdf = x / (display_scale * RENDER_SCALE)
    y_pdf = y / (display_scale * RENDER_SCALE)
    idx = len(points) + 1
    points.append({'name': f'point_{idx}', 'x_pdf': round(x_pdf,1), 'y_pdf': round(y_pdf,1), 'lat': None, 'lng': None})
    redraw()
    build_form()

canvas.on_mouse_down(handle_click)

remove_btn = widgets.Button(description='Remover último ponto')
def on_remove(b):
    if points:
        points.pop()
        redraw()
        build_form()
remove_btn.on_click(on_remove)

form_box = widgets.VBox()

def build_form():
    rows = []
    for i, p in enumerate(points):
        name_w = widgets.Text(value=p['name'], description=f'#{i+1} nome:', style={'description_width':'80px'}, layout=widgets.Layout(width='300px'))
        lat_w = widgets.Text(value='' if p['lat'] is None else str(p['lat']), description='lat:', layout=widgets.Layout(width='200px'))
        lng_w = widgets.Text(value='' if p['lng'] is None else str(p['lng']), description='lng:', layout=widgets.Layout(width='200px'))
        coord_label = widgets.Label(value=f"x_pdf={p['x_pdf']}  y_pdf={p['y_pdf']}")

        def make_handler(i, field):
            def handler(change):
                points[i][field] = change['new']
            return handler
        name_w.observe(make_handler(i,'name'), names='value')
        lat_w.observe(make_handler(i,'lat'), names='value')
        lng_w.observe(make_handler(i,'lng'), names='value')

        rows.append(widgets.HBox([name_w, lat_w, lng_w, coord_label]))
    form_box.children = rows

display(widgets.VBox([canvas, remove_btn]))
display(form_box)
build_form()
print('Clique no mapa acima para adicionar pontos. Preencha nome/lat/lng nos campos que aparecem embaixo.')

## 4. Validar e exportar CSV
Roda essa célula depois de preencher todos os lat/lng. Ela avisa se algum campo ficou vazio ou não-numérico, e só então gera o CSV pra download.

In [ ]:
errors = []
clean_rows = []
for i, p in enumerate(points):
    try:
        lat = float(p['lat'])
        lng = float(p['lng'])
    except (TypeError, ValueError):
        errors.append(f"Ponto #{i+1} ({p['name']}): lat/lng vazio ou inválido -> lat={p['lat']!r} lng={p['lng']!r}")
        continue
    clean_rows.append({
        'name': p['name'],
        'x_pdf': p['x_pdf'],
        'y_pdf': p['y_pdf'],
        'lat': lat,
        'lng': lng,
    })

if errors:
    print('ATENCAO - corrija antes de exportar:')
    for e in errors:
        print(' -', e)
else:
    print(f'{len(clean_rows)} pontos validos. Gerando CSV...')
    out_name = 'gcp_points.csv'
    with open(out_name, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['name','x_pdf','y_pdf','lat','lng'])
        writer.writeheader()
        writer.writerows(clean_rows)
    print('Salvo:', out_name)
    for r in clean_rows:
        print(r)
    files.download(out_name)

## 5. (Opcional) Conferencia visual rapida
Mostra a imagem com os pontos numerados de novo, tamanho grande, para conferir antes de mandar pro Claude.

In [ ]:
from PIL import ImageDraw
check_img = img.copy()
draw = ImageDraw.Draw(check_img)
for i, p in enumerate(points):
    x = p['x_pdf'] * RENDER_SCALE
    y = p['y_pdf'] * RENDER_SCALE
    r = 10
    draw.ellipse([x-r,y-r,x+r,y+r], outline=(255,0,0), width=4)
    draw.text((x+12,y-12), str(i+1), fill=(255,0,0))
check_img.save('gcp_check.png')
check_img